In [2]:

import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import shufflenet_v2_x1_0
from sklearn.model_selection import train_test_split

# FX Quantization imports 
from torch.ao.quantization import QConfig, QConfigMapping
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx



# Repro & Backend setup

torch.backends.quantized.engine = "fbgemm"  # x86 CPU backend for INT8
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")  # PTQ/eval run on CPU



# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Class names (first 10):", dataset.classes[:10], "...")

# Stratified split (85/10/5) 
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)  # 0.10/0.05
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")



# Load your trained FP32 ShuffleNetV2

ckpt_path = "/home/h5/sapi279d/shufflenetv2_weights.pth"
state_dict = torch.load(ckpt_path, map_location="cpu")

# Your weights were saved from a compiled/wrapped model → strip the prefix
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

num_classes = len(dataset.classes)  # 200 in your case
model_fp32 = shufflenet_v2_x1_0(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, num_classes)
model_fp32.load_state_dict(new_state_dict, strict=True)
model_fp32.eval()
print("FP32 ShuffleNetV2 loaded")



# Per-channel Symmetric PTQ via FX Graph

# Build an explicit per-channel symmetric qconfig for weights
per_channel_qconfig = QConfig(
    activation=MinMaxObserver.with_args(dtype=torch.quint8, qscheme=torch.per_tensor_affine),
    weight=PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric)
)
qconfig_mapping = QConfigMapping().set_global(per_channel_qconfig)

# Example inputs for FX (shape must match model)
example_inputs = (torch.randn(1, 3, 224, 224),)

# Prepare (FX inserts Quantize/DeQuantize)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

# Calibration: run a few batches through the prepared model
def calibrate_fx(model, loader, num_batches=20):
    model.eval()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            model(images)

calibrate_fx(prepared, val_loader, num_batches=20)  # you can also use train_loader
print("Calibration done (FX, per-channel)")

# Convert to INT8 (FX)
model_int8 = convert_fx(prepared).to("cpu").eval()
print("Converted to per-channel symmetric INT8 (FX)")



# Evaluation

def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)  # float inputs OK; FX handles Q/DQ internally
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_int8 = evaluate_model(model_int8, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Per-channel Sym, FX): {acc_int8:.2f}%")



# Model size & Runtime benchmarks

def get_model_size(model, filename="__temp__.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    ms_per_batch = (total_time / num_batches) * 1000.0
    ms_per_image = (total_time / total_images) * 1000.0
    return ms_per_batch, ms_per_image

fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)
print(f"FP32 size: {fp32_size:.2f} MB | INT8 size: {int8_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_int8, img_ms_int8 = benchmark(model_int8, test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {batch_ms_int8:.2f} ms/batch | {img_ms_int8:.2f} ms/image")

throughput_fp32 = 1000.0 / img_ms_fp32 if img_ms_fp32 > 0 else float('inf')
throughput_int8 = 1000.0 / img_ms_int8 if img_ms_int8 > 0 else float('inf')
print(f"FP32 Throughput: {throughput_fp32:.2f} img/s")
print(f"INT8 Throughput: {throughput_int8:.2f} img/s")





Total images: 100000
Number of classes: 200
Class names (first 10): ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
Train: 85000, Val: 10000, Test: 5000
DataLoaders ready ✅
✅ FP32 ShuffleNetV2 loaded


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/cuda/__init__.py:141: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 802: system not yet initialized (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


Calibration done ✅ (FX, per-channel)
Converted to per-channel symmetric INT8 (FX) ✅
FP32 Accuracy: 49.98%
INT8 Accuracy (Per-channel Sym, FX): 46.44%
FP32 size: 6.00 MB | INT8 size: 1.72 MB
FP32 Inference: 644.64 ms/batch | 5.04 ms/image
INT8 Inference: 2662.83 ms/batch | 20.80 ms/image
FP32 Throughput: 198.56 img/s
INT8 Throughput: 48.07 img/s
💾 Saved quantized model: shufflenet_perchannel_symmetric_fx.pth
